# GRU v2·v3 확률 보정과 경보 성능 비교
**GPU와 GRU 재학습 없이 실행합니다.** Google Drive의 두 실험에서 `summary.json`, `oof_predictions.parquet`만 읽습니다. 원본 ZIP·학습 입력·GRU 체크포인트는 필요하지 않습니다.

보정 없음(raw), Platt, Isotonic을 비교합니다. 각 기존 GRU 검증 fold 안의 환자를 3개 그룹으로 나누고 **보정 학습 / 경보 기준 설정 / 평가** 역할을 번갈아 맡깁니다. 두 실험은 동일한 환자 그룹을 사용합니다.

GRU의 epoch 및 하이퍼파라미터 선택에는 이미 개발 데이터가 사용됐습니다. 따라서 이 결과는 **개발 단계 비교**이며 독립적인 최종 Test 성능이나 완전한 중첩 교차검증 결과가 아닙니다. Test 파일을 읽거나 평가하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
RESULTS_ROOT = Path("/content/drive/MyDrive/SEWS_GRU/results")
V2_NAME = "gru_v2_lr0003"
V3_NAME = "gru_v3_lr0003_no_weight"
for name in [V2_NAME, V3_NAME]:
    for filename in ["summary.json", "oof_predictions.parquet"]:
        path = RESULTS_ROOT / name / filename
        assert path.is_file(), f"파일이 없습니다. 실험 폴더 이름을 확인하세요: {path}"
print("두 실험의 결과 파일 확인 완료")


## 1. 비교용 환경 준비
Python 3.10과 평가 라이브러리만 설치합니다. 기존 학습 노트북의 변수나 GPU 환경에 의존하지 않습니다. 학습 코드가 있는 폴더와 별도의 폴더를 사용합니다.

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/sogang-icu-sews/ICU-SEWS.git"
BRANCH = "codex/gru-colab"
REPO = Path("/content/ICU-SEWS-GRU-eval")
def run(command):
    return subprocess.run([str(x) for x in command], check=True)

if not REPO.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO])
else:
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL, "저장소 주소가 다릅니다."
    run(["git", "-C", REPO, "pull", "--ff-only"])
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = [sys.executable, "-m", "uv"]
PY = Path("/content/sews-eval-py310/bin/python")
if not PY.exists():
    run(UV + ["venv", "--python", "3.10", str(PY.parent.parent)])
run(UV + ["pip", "install", "--python", str(PY), "-r", str(REPO / "requirements-gru-eval.txt")])
run([PY, "-c", "import sys,sklearn; assert sys.version_info[:2]==(3,10); print(sys.version); print('sklearn',sklearn.__version__)"])


## 2. 비교 실행
각 모델·보정 방법에 대해 세 가지 경보 정책을 평가합니다.

- **row_specificity95**: 경보 기준 설정 그룹에서 시간 행 단위 특이도 ≥95%.
- **patient_false_alarm10**: 경보 기준 설정 그룹의 비패혈증 환자 중 한 번이라도 경보를 받는 비율 ≤10%.
- **max_utility**: 경보 기준 설정 그룹에서 PhysioNet 정규화 Utility를 최대화.

이 수치는 비교를 위한 사전 설정이며 임상 운영 기준이 아닙니다. 평가 그룹에서도 정확히 같은 특이도·오경보율을 얻는다는 보장은 없습니다. 실제 측정된 값을 함께 확인하세요.

실행할 때마다 새 결과 폴더가 생깁니다. 실패한 실행을 이어가지는 않으며, 재실행하면 보정·평가 계산만 다시 합니다.

In [ ]:
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

CACHE = Path("/content/sews-gru-comparison-inputs")
for label, name in [("v2", V2_NAME), ("v3", V3_NAME)]:
    destination = CACHE / label
    destination.mkdir(parents=True, exist_ok=True)
    for filename in ["summary.json", "oof_predictions.parquet"]:
        shutil.copyfile(RESULTS_ROOT / name / filename, destination / filename)

stamp = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")
OUT = Path("/content/drive/MyDrive/SEWS_GRU/comparisons") / f"v2_v3_{stamp}"
run([PY, "-u", REPO / "scripts/compare_gru.py",
     "--v2", CACHE / "v2", "--v3", CACHE / "v3", "--out", OUT, "--seed", "42"])
print("결과 저장 위치:", OUT)


## 3. 확률 보정 비교
**fold=-1은 모든 평가 예측을 합친 결과**이며 fold별 평균과 다릅니다. 아래 두 표를 구분해 보세요.

- Brier·log loss·ECE: 낮을수록 좋습니다. ECE는 10개 동일 너비 구간 기준이며 단독 선택 지표로 쓰지 않습니다.
- baseline_Brier: 평가 환자의 정답을 사용하지 않고, 해당 보정 학습 그룹의 양성 비율을 모든 환자에게 예측한 기준.
- Brier_skill: 기준 대비 개선 비율. 0이면 기준과 같고, 음수면 기준보다 나쁩니다.
- mean_prediction과 prevalence: 평균 예측값과 실제 양성 비율.
- 보정 후 AUPRC가 일부 바뀔 수 있습니다. Isotonic은 동점을 만들고, 여러 보정기를 합친 평가에서는 전체 순위도 달라질 수 있습니다.

In [ ]:
import json
import pandas as pd
from pathlib import Path

# 런타임이 초기화된 후 결과만 볼 때는 Drive 연결 후 이 셀부터 실행해도 됩니다.
if "OUT" not in globals() or not (OUT / "metadata.json").exists():
    completed = sorted(Path("/content/drive/MyDrive/SEWS_GRU/comparisons").glob("*/metadata.json"))
    assert completed, "완료된 비교 결과가 없습니다."
    OUT = completed[-1].parent
print("읽는 결과:", OUT)
print(json.loads((OUT / "metadata.json").read_text())["limitation"])

prob = pd.read_csv(OUT / "probability_metrics.csv")
columns = ["model", "method", "AUROC", "AUPRC", "Brier", "baseline_Brier",
           "Brier_skill", "log_loss", "ECE_10_equal_width", "prevalence", "mean_prediction"]
print("① 전체 평가 예측을 합친 결과")
display(prob[(prob.fold == -1) & (prob.scope == "all")][columns].round(5))

print("② 기존 5개 fold별 성능의 평균·표준편차")
display(prob[(prob.fold >= 0) & (prob.scope == "all")]
        .groupby(["model", "method"])[["AUROC", "AUPRC", "Brier", "Brier_skill"]]
        .agg(["mean", "std"]).round(5))

print("③ 병원별·60시간 이내 결과")
display(prob[(prob.fold == -1) & (prob.scope != "all")]
        [["model", "method", "scope", "AUROC", "AUPRC", "Brier", "Brier_skill"]].round(5))


## 4. 경보 성능 비교
모든 비율은 0~1입니다. 예: 0.10 = 10%.

- row_sensitivity: 양성 시간 행 중 경보가 울린 비율.
- row_PPV: 경보가 울린 시간 행 중 공식 라벨이 양성인 비율. 발병 6시간보다 이른 유용한 경보도 행 단위로는 음성일 수 있으므로 Utility·조기 감지와 함께 봅니다.
- nonsepsis_patient_false_alarm_rate: 기록 전체가 비패혈증인 환자 중 경보를 한 번 이상 받은 비율.
- early_recall_6to12h: 추정 발병 6~12시간 전 구간에서 한 번 이상 경보가 나온 패혈증 환자 비율.
- pre_onset_recall_0to12h: 추정 발병 12시간 전부터 발병 직전까지 한 번 이상 경보가 나온 비율.
- median_lead_hours_detected: 위 0~12시간 구간에서 감지된 환자에 한해 계산한 최초 유효 경보의 중앙 선행 시간.
- Utility: PhysioNet 정규화 유틸리티. 높을수록 좋고, 0은 경보를 전혀 울리지 않는 기준입니다.

발병 시각은 첫 양성 라벨 +6시간으로 역산합니다. 첫 행부터 양성인 환자는 시각을 알기 어려워 **조기 감지·선행 시간 지표에서 제외**하고 제외 수를 저장합니다. Utility는 공식 정의대로 전체 환자를 사용합니다.
매 시간 임계값을 비교하며 경보 억제 시간(cooldown)은 없습니다. 경보 시간 수를 실제 알림 발생 건수로 해석하지 마세요.

In [ ]:
alarms = pd.read_csv(OUT / "alarm_metrics.csv")
alarm_columns = ["model", "method", "row_sensitivity", "row_PPV", "row_specificity",
                 "nonsepsis_patient_false_alarm_rate", "early_recall_6to12h",
                 "pre_onset_recall_0to12h", "median_lead_hours_detected", "Utility"]
for policy, title in [
    ("row_specificity95", "① 시간별 특이도 95% 목표"),
    ("patient_false_alarm10", "② 비패혈증 환자 오경보율 10% 목표"),
    ("max_utility", "③ Utility 최적화 기준"),
]:
    print(title)
    display(alarms[(alarms.fold == -1) & (alarms.policy == policy)]
            [alarm_columns].round(5))

print("실제 경보 부담과 조기 평가 대상 수")
display(alarms[(alarms.fold == -1) & (alarms.policy == "patient_false_alarm10")]
        [["model", "method", "alarm_hours_per100h", "false_alarm_hours_per100_nonsepsis_h",
          "timing_eligible_patients", "timing_excluded_starts_positive"]].round(4))


## 5. 보정 곡선과 저장 결과
점들이 대각선에 가까울수록 해당 구간의 평균 예측과 실제 양성 비율이 비슷합니다. 전체 그림과 0~10% 확대 그림을 함께 봅니다.

학습·평가에 사용한 환자 역할, 모델별 보정 예측, fold/그룹별 임계값, 보정기를 모두 저장합니다. 저장된 보정기와 임계값은 **이번 분할의 평가용**이며 바로 서비스에 연결할 최종 보정기·단일 경보 기준이 아닙니다.

같은 순서를 유지하는 Platt 보정은 순위 기반 경보 정책의 결과를 거의 바꾸지 않을 수 있습니다. 보정의 목적은 확률 해석을 개선하는 것이며, 경보 성능 개선을 보장하지 않습니다.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUT / "calibration.png")))
display(Image(filename=str(OUT / "calibration_zoom.png")))
print("저장 폴더:", OUT)
print("공유하면 되는 요약 파일:")
for name in ["probability_metrics.csv", "alarm_metrics.csv",
             "probability_fold_summary.csv", "alarm_fold_summary.csv"]:
    print(" -", name)
